# Uji Validitas dan Reliabilitas – Topik 2: Kebiasaan Digital – 3 TI A

Notebook ini mengikuti tahapan pada Modul Pengujian Validitas dan Reliabilitas Instrumen. Jalankan sel secara berurutan.

## Langkah 1. Persiapan lingkungan dan impor pustaka

In [ ]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files


## Langkah 2. Unggah berkas data Excel

In [ ]:
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)


## Langkah 3. Membaca data dan memilih kelas 3 TI A

File sumber memiliki kolom `Cap waktu` dan `Kelas`, sehingga butir dimulai dari kolom ketiga. Data difilter agar hanya kelas 3 TI A yang dianalisis.

In [ ]:
df_awal = pd.read_excel(nama_file)
df = df_awal[df_awal["Kelas"].astype(str).str.strip() == "3 TI A"].copy()

JUMLAH_KOLOM_DEMOGRAFI = 2  # Cap waktu dan Kelas
butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]

print("Jumlah responden kelas 3 TI A:", len(df))
print("Jumlah butir:", butir.shape[1])
print("Data kosong:", int(butir.isna().sum().sum()))


## Langkah 4. Mengubah jawaban Likert menjadi angka

In [ ]:
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5
}

num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
display(num.head())


## Langkah 5. Mengelompokkan butir per dimensi

In [ ]:
dimensi = {}
for kolom in butir.columns:
    cocok = re.match(r"^(.*?) \[", kolom)
    if cocok is None:
        raise ValueError(f"Format judul kolom tidak sesuai: {kolom}")
    nama = cocok.group(1)
    dimensi.setdefault(nama, []).append(kolom)

for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")


## Langkah 6. Uji validitas (korelasi Pearson butir-total)

In [ ]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")

hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid"
        })

tabel_validitas = pd.DataFrame(hasil_validitas)
display(tabel_validitas)


## Langkah 7. Uji reliabilitas (Cronbach's Alpha)

In [ ]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []

for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3),
        "Kategori": kategori(a)
    })
    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3)
            })

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)


## Langkah 8. Analisis deskriptif dan mengunduh hasil

In [ ]:
hasil_deskriptif = []
for nama, daftar in dimensi.items():
    skor_dimensi = num[daftar].sum(axis=1)
    hasil_deskriptif.append({
        "Dimensi": nama,
        "Jumlah responden": len(skor_dimensi),
        "Skor minimum": skor_dimensi.min(),
        "Skor maksimum": skor_dimensi.max(),
        "Rata-rata": skor_dimensi.mean(),
        "Standar deviasi": skor_dimensi.std(),
        "Skewness": stats.skew(skor_dimensi, bias=False),
        "Kurtosis": stats.kurtosis(skor_dimensi, bias=False)
    })

tabel_deskriptif = pd.DataFrame(hasil_deskriptif)
display(tabel_deskriptif)

gabung = tabel_validitas.merge(
    tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left"
)

with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as writer:
    gabung.to_excel(writer, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(writer, sheet_name="Reliabilitas", index=False)
    tabel_deskriptif.to_excel(writer, sheet_name="Deskriptif", index=False)

files.download("hasil_uji_validitas_reliabilitas.xlsx")


## Catatan interpretasi

- Butir dinyatakan valid jika r hitung > r tabel, sesuai kriteria modul.
- Reliabilitas dinyatakan memenuhi kriteria jika Cronbach's Alpha minimal 0,70.
- Alpha jika butir dihapus dibandingkan dengan alpha dimensi. Keputusan menghapus butir tidak boleh hanya berdasarkan angka; isi butir tetap harus dipertimbangkan.
- Jika terdapat data kosong atau sel gagal dikonversi, periksa data sebelum menarik kesimpulan.